# Phase 6.5B — Live PDF development pilot
One table target, actual PDF parsing and local hybrid RAG, followed by optional new DeepSeek matching and extraction. This is a development integration test, not held-out evaluation. Default is zero new paid calls. No old replay ZIPs or reference labels are required. Library is read-only. Use a development SAP and verify the physical SAP body range excludes appended reference shells. Only enable LIVE after explicitly approving up to two new completion attempts and confirming permission to send retrieved SAP excerpts to the provider.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, json, shutil, time
from datetime import datetime, timezone
REPO='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
PIN='3544f2c0d557d12285336984a456ec84bd357ad2'
ROOT=Path('/content/tfl-phase65b-pinned')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO,str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',PIN],check=True)
COMMIT=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert COMMIT==PIN
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in (SRC/'tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Pinned code:',COMMIT)
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_unified_pipeline.py')],check=True)

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive',timeout_ms=180000)
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
assert BASE.is_dir(), f'Missing project directory: {BASE}'
LIBRARY=ROOT/'generic_template_library'
assert LIBRARY.is_dir()
print('PDF paths relative to project folder:')
for p in sorted(BASE.rglob('*.pdf')):
    print(p.relative_to(BASE))


## Configure one development case
Select CASE_ID (default 2). The notebook locates an exact historical PDF by SHA256 in the existing Drive project directory, so filenames do not matter. It does not require the old experiment ZIPs. If no matching PDF exists, restore the original PDF, not a replay archive. The historical page ranges below are candidate ranges, not newly verified boundaries. Inspect the original PDF and the displayed page excerpts, adjust if necessary, and explicitly confirm the body interval excludes reference shells before preparing requests. No model calls occur here.

In [ ]:
CASE_ID=2 #@param {type:"integer"}
from tfl_agent.agent.unified_pipeline import sha_file
CASES={
    1:('ARV-471-BC-201','Summary of Protocol Deviation','ac6ebc7497c8ea2d5ffd31834e02f95206884f7ac54e69e54a3cf7f9f4cd8caf',10,30),
    2:('ARV-471-BC-201','Summary of Radiographic Response per mRECIST during Cycle 6','ac6ebc7497c8ea2d5ffd31834e02f95206884f7ac54e69e54a3cf7f9f4cd8caf',10,30),
    3:('OMO-103-01','Summary of Demographic and Baseline Information','1aa3f12d5e1f3fba31e789edc9911b3dfb4622e3940c79acfab56a074b04255a',8,29),
    4:('OMO-103-01','Frequency of Autoimmune Values','1aa3f12d5e1f3fba31e789edc9911b3dfb4622e3940c79acfab56a074b04255a',8,29),
    5:('SP11631','Adverse Events – Safety Analysis Set','513b744f284eca557817ab39dd036972ff6b1e5e12ff924b71a56646168c55f7',5,27),
    6:('SP11631','TEAEs by System Organ Class and Preferred Term and NCI CTCAE V4.03 Grade - Safety Analysis Set','513b744f284eca557817ab39dd036972ff6b1e5e12ff924b71a56646168c55f7',5,27)}
assert CASE_ID in CASES,'Choose CASE_ID 1 through 6'
STUDY,TARGET_TITLE,EXPECTED_SHA,SUGGESTED_FIRST,SUGGESTED_LAST=CASES[CASE_ID]
matches=[p for p in sorted(BASE.rglob('*')) if p.is_file() and p.suffix.lower()=='.pdf' and sha_file(p)==EXPECTED_SHA]
if not matches:
    raise FileNotFoundError(f'Original SAP for {STUDY} not found under {BASE}. Restore the PDF with SHA256 {EXPECTED_SHA}; old experiment ZIPs are not a substitute.')
SAP=matches[0].resolve()
assert SAP.is_relative_to(BASE.resolve())
SAP_RELATIVE_PATH=str(SAP.relative_to(BASE.resolve()))
print('Study:',STUDY)
print('Target:',TARGET_TITLE)
print('Verified original PDF:',SAP_RELATIVE_PATH)
print('Identical copies:',len(matches))
print('Candidate body pages:',SUGGESTED_FIRST,SUGGESTED_LAST)
import pymupdf
with pymupdf.open(SAP) as doc:
    PDF_PAGE_COUNT=len(doc)
    print('Physical PDF pages:',PDF_PAGE_COUNT)
    for p in sorted({SUGGESTED_FIRST-1,SUGGESTED_FIRST,SUGGESTED_LAST,SUGGESTED_LAST+1}):
        if 1<=p<=PDF_PAGE_COUNT:
            t=doc[p-1].get_text()
            print(f'\n--- Physical page {p} ---\n{t[:700]}\n...\n{t[-400:]}')

## Verify SAP body and prepare
Leave the override values at 0 to use the suggested historical range. If inspection shows a different boundary, enter actual inclusive physical page numbers. Confirm only after checking that the complete selected interval is SAP body and does not contain appended reference shells; excerpts are a convenience, not automatic verification.

In [ ]:
BODY_FIRST_OVERRIDE=0 #@param {type:"integer"}
BODY_LAST_OVERRIDE=0 #@param {type:"integer"}
BODY_RANGE_CONFIRMED=False #@param {type:"boolean"}
BODY_FIRST=BODY_FIRST_OVERRIDE or SUGGESTED_FIRST
BODY_LAST=BODY_LAST_OVERRIDE or SUGGESTED_LAST
if not 1<=BODY_FIRST<=BODY_LAST<=PDF_PAGE_COUNT:
    raise ValueError('Invalid physical PDF body range.')
if not BODY_RANGE_CONFIRMED:
    raise ValueError('Inspect the PDF body range, then set BODY_RANGE_CONFIRMED=True. No LLM calls were made.')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase65B_Live_Pilot')/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
DEST=BASE/'Phase65B_Live_Pilot_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
CACHE=BASE/'Phase65B_Live_Cache'/'Pilot01'
from tfl_agent import TFLAgent
from tfl_agent.agent.unified_pipeline import ResponseExecutor, finish_prepared, summarize, write_json, read_json
import torch
from tfl_agent.tools.sap_hybrid import LocalEncoder
start=time.perf_counter()
encoder=LocalEncoder(device='cuda' if torch.cuda.is_available() else 'cpu')
agent=TFLAgent(library_path=LIBRARY,model='deepseek-flash',encoder=encoder)
plan=agent.prepare(SAP,TARGET_TITLE,body_pages=(BODY_FIRST,BODY_LAST))
prepare_seconds=time.perf_counter()-start
assert plan['mode']=='fresh_pdf'
assert plan['retrieval']['primary_ids']
write_json(OUT/'Prepared_Plan.json',plan)
identity={'commit':COMMIT,'sap_sha256':plan['sap_sha256'],'target':plan['target'],
          'body_pages':[BODY_FIRST,BODY_LAST],'matching_fingerprint':plan['matching_request']['fingerprint'],
          'specification_fingerprint':plan['specification_request']['fingerprint']}
write_json(OUT/'Pilot_Identity.json',identity)
print(json.dumps({'identity':identity,'blocks':plan['retrieval']['block_count'],
                  'chunks':plan['retrieval']['chunk_count'],
                  'primary':len(plan['retrieval']['primary_ids']),
                  'supplementary':len(plan['retrieval']['supplementary_ids']),
                  'prepare_seconds':round(prepare_seconds,2)},indent=2))
for c in plan['source_chunks']:
    print(c['chunk_id'],c['pages'],c['section'],c['text'][:220])
saved=Path(shutil.make_archive(str(DEST/RUN_ID)+'_prepared','zip',root_dir=OUT))
print('Prepared-only ZIP:',saved)

## Finish the prepared plan
Run the next cell with LIVE=False for a zero-call smoke test. It is expected to report not_attempted on cache misses. For the actual pilot, opt in by changing LIVE=True and rerunning this cell (not the preparation cell unless necessary). A positive cap is fixed at two, not two per rerun. The model requests are content-addressed; changed SAP, target, code or fingerprints cannot reuse this pilot ledger. API errors consume a reserved attempt and stop further new calls in the current executor. No automatic retry. Do not reset the ledger. The DeepSeek key must be stored in Colab Secrets as DEEPSEEK_API_KEY. Never print the key. Do not run multiple runtimes against this ledger.

In [ ]:
LIVE=False #@param {type:"boolean"}
executor=None
try:
    if LIVE:
        from google.colab import userdata
        key=userdata.get('DEEPSEEK_API_KEY').strip()
        if not key or any(c.isspace() for c in key):
            raise ValueError('Invalid DeepSeek secret format')
        CACHE.mkdir(parents=True,exist_ok=True)
        manifest=CACHE/'Pilot_Identity.json'
        if manifest.exists():
            if read_json(manifest)!=identity:
                raise ValueError('Pilot identity changed; do not reset or change this ledger.')
        else:
            if (CACHE/'Budget.json').exists() or list(CACHE.glob('*.attempt.json')):
                raise ValueError('Existing ledger without identity; inspect before running.')
            with manifest.open('x',encoding='utf-8') as f:
                json.dump(identity,f,ensure_ascii=False,indent=2)
        executor=ResponseExecutor(cache_dir=CACHE,authorized=True,max_new_attempts=2,api_key=key)
        del key
    else:
        executor=ResponseExecutor()
    t=time.perf_counter()
    result=finish_prepared(plan,executor)
    finish_seconds=time.perf_counter()-t
    assert result['mode']=='fresh_pdf'
    assert not result['generation_authorized'] and result['confirmed_template'] is None
    assert result['new_attempts']<= (2 if LIVE else 0)
    summary=summarize([result])
    summary.update(pilot_type='development_live' if LIVE else 'fresh_dry_run',
                   identity=identity,prepare_seconds=round(prepare_seconds,3),
                   finish_seconds=round(finish_seconds,3),
                   stage_status=result['stage_status'],
                   recommended_template=result['recommended_template'],
                   proposed_preset=result['proposed_preset'],
                   required_field_gaps=len(result['required_field_gaps']),
                   candidate_fields=sum(f['status']=='candidate' for f in result['specification']['fields'].values()),
                   retrieval_counts={k:len(plan['retrieval'][k]) for k in ('primary_ids','supplementary_ids')},
                   response_audit={k:{q:v.get(q) for q in ('status','cache_source','finish_reason','usage','reason','error_type','http_status')} for k,v in result['responses'].items()},
                   ledger_attempts=len(list(CACHE.glob('*.attempt.json'))) if LIVE else None,
                   held_out=False)
    write_json(OUT/'Fresh_PDF_Result.json',result)
    write_json(OUT/'Summary.json',summary)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
except Exception as exc:
    write_json(OUT/'Failure.json',{'error_type':type(exc).__name__,'live':LIVE,
                                    'new_attempts':executor.new_attempts if executor else 0})
    print('Pilot failed:',type(exc).__name__,'— inspect saved artifacts before retrying.')
    raise
finally:
    suffix='_live' if LIVE else '_dry'
    saved=Path(shutil.make_archive(str(DEST/RUN_ID)+suffix,'zip',root_dir=OUT))
    print('Saved:',saved)
    print('Share only if authorized to disclose SAP excerpts and model responses.')

## Interpretation
A successful fresh candidate pipeline requires both complete parsed responses and a non-null candidate template. Report matching protocol validity separately. Claims and presets are not semantically approved; generation remains unauthorized. Zero-call preparation is not a live inference pass. A one-case live test is not an accuracy estimate. Return the live ZIP (or the dry ZIP if preparation or inference failed), not the secret or cache directory. Phase 6.5C will freeze the policy and use independent held-out labels only at scoring time.